In [1]:
import tensorflow as tf
import TensorSlider as ts
import keras
import numpy as np
import DataPrep


2025-02-08 14:45:45.250227: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739022345.262537  116741 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739022345.266152  116741 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-08 14:45:45.283528: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Get Datasets

In [3]:
tfrecordpath = "../Data/tfrecords/"

windowsize = 150
lookahead = 5
batch_size = 100

coins = ["BTCUSD_PERP", "ETHUSD_PERP", "ADAUSD_PERP", "SOLUSD_PERP", "BNBUSDT_PERP", "LINKUSD_PERP", "TRXUSD_PERP", "XLMUSD_PERP","DOTUSD_PERP"]
datasets = DataPrep.getAllSliders(coins, windowsize, lookahead, batch_size, tfrecordpath)
valdataset = datasets.pop(-1)

I0000 00:00:1739022348.058103  116741 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5592 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:05:00.0, compute capability: 8.6


## Combine Datasets

In [4]:
import keras
import os

def load_model(name, optimizer):
    """
    Load model and latest checkpoint (if applicable). Returns model, and last epoch that was trained.
    If no checkpoints present, either creates the checkpoint folder or trains directly on the saved model.
    """
    folder = "models/" + name + "/"
    # Load model
    model = keras.models.load_model(folder + "model.keras")

    checkpoint = tf.train.Checkpoint(optimizer=optimizer, model=model)
    manager = tf.train.CheckpointManager(checkpoint=checkpoint, directory=folder, max_to_keep=3)
    manager.restore_or_initialize()

    return model, manager

class saveEachEpoch(tf.keras.callbacks.Callback):
    """
    custom callback for saving checkpoints because of course we need to do this on our own
    """

    def __init__(self, checkpointmanager):
        super().__init__()
        # no idea if we want to or need to super this
        try:
            self.lastEpoch = checkpointmanager.latest_checkpoint.split("-")[-1]
        except Exception as e:
            print(e)
            self.lastEpoch = 0

        self.checkpointManager = checkpointmanager
        print("Model was trained for " + str(self.lastEpoch) + " epochs before.")

    def on_epoch_end(self, epoch, logs):
        """
        Create a checkpoint and save.
        """
        print(f"Epoch {epoch} ended")
        # Increment which epoch this is
        self.lastEpoch += 1
        # Save weights
        self.checkpointManager.save(checkpoint_number=epoch)


In [5]:
#zip the different dataset sources
zipped = tf.data.Dataset.zip(datasets=tuple(datasets))
def combineZippedBatches(*zipped):
    batchshape = zipped[0][0]
    # Create first tensors to concat the rest
    data = zipped[0][0]
    label = zipped[0][1]
    for i in range(1, len(zipped)): # iterate over each remaining pair
        data = tf.concat([data, zipped[i][0]], axis=0)
        label = tf.concat([label, zipped[i][1]], axis=0)

    return data, label

# combine batches into one megabatch
batchTogether = zipped.map(combineZippedBatches)

# Prefetch and create labels
training = batchTogether.map(DataPrep.createLabelsBatch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
validation = valdataset.map(DataPrep.createLabelsBatch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [ ]:
modelName = "bigmodel"

tensorboard = keras.callbacks.TensorBoard(
                                                log_dir=f"models/{modelName}/logs",
                                                histogram_freq=100,
                                                write_graph=True,
                                                write_images=False,
                                                write_steps_per_second=True,
                                                update_freq="batch",
                                                #profile_batch = '70,100',
                                                embeddings_freq=0,
                                                embeddings_metadata=None,
                                            )
optimizer = keras.optimizers.Adam(amsgrad=True, clipnorm=1)

model, manager = load_model(modelName, optimizer)

model.compile(loss=keras.losses.mean_squared_error,
                  optimizer=optimizer,
                  # For the metrics, always have AT LEAST these two
                  metrics=["MeanAbsolutePercentageError", "MeanSquaredError"])

history = model.fit(training, epochs=20, verbose=0, validation_data=validation, callbacks=[saveEachEpoch(manager), tensorboard])

Model was trained for 5 epochs before.


2025-02-08 14:46:03.583887: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:370] TFRecordDataset `buffer_size` is unspecified, default to 262144
